# 🧠 Tactile SDF Reconstruction — Google Colab

This notebook trains a **PointNet + SIREN** model using sparse tactile contacts.

**Data Sources:**
- **Grasp Data + SDF Cache:** `jack635/grasp-dataset-curated` (Hugging Face)
- **GT Meshes:** Objaverse API (50 objects)

In [ ]:
# @title 🛠️ Step 1: Setup & Dependencies
!pip install -q trimesh rtree plotly scikit-image scikit-learn tqdm huggingface_hub torch

In [ ]:
# @title 📂 Step 2: Download Code & Data from HF
import os
import json
from huggingface_hub import snapshot_download

ROOT = "/content/tactile_project"
os.makedirs(ROOT, exist_ok=True)

# 1. Clone the tactile-sdf code
if not os.path.exists(f'{ROOT}/code'):
    !git clone https://github.com/635jack/tactile-sdf.git {ROOT}/code
else:
    !cd {ROOT}/code && git pull

# 2. Download full dataset + precomputed SDF cache from HF
repo_id = "jack635/grasp-dataset-curated"
if not os.path.exists(f'{ROOT}/grasp-dataset/dataset_index.json'):
    print(f"📥 Downloading dataset + SDF cache from {repo_id}...")
    snapshot_download(repo_id=repo_id, repo_type="dataset", local_dir=f"{ROOT}/grasp-dataset")
    print("✅ Download complete")
else:
    print("✅ Dataset already downloaded")

# Check SDF cache
sdf_cache_dir = f"{ROOT}/grasp-dataset/sdf_cache"
n_sdf = len(os.listdir(sdf_cache_dir)) if os.path.exists(sdf_cache_dir) else 0
print(f"📦 SDF cache: {n_sdf}/42 objects ready")

In [ ]:
# @title 🚀 Step 3: Train!
# GPU is used automatically if enabled (Runtime > Change runtime type > T4 GPU)
import subprocess, sys

cmd = [
    sys.executable, f"{ROOT}/code/train.py",
    "--epochs", "300",
    "--batch_size", "16",
    "--dataset_dir", f"{ROOT}/grasp-dataset",
    "--sdf_cache_dir", f"{ROOT}/grasp-dataset/sdf_cache",
    "--output_dir", f"{ROOT}/runs",
    "--eval_every", "10",
    "--save_every", "50",
]
print("🚀 Starting training...")
!{sys.executable} {ROOT}/code/train.py \
    --epochs 300 \
    --batch_size 16 \
    --dataset_dir {ROOT}/grasp-dataset \
    --sdf_cache_dir {ROOT}/grasp-dataset/sdf_cache \
    --output_dir {ROOT}/runs \
    --eval_every 10 \
    --save_every 50

In [ ]:
# @title 📊 Step 4: Visualize Results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

runs = sorted(glob.glob(f'{ROOT}/runs/*'))
if runs:
    latest_run = runs[-1]
    print(f"Latest run: {latest_run}")
    img_path = os.path.join(latest_run, 'training_curves.png')
    if os.path.exists(img_path):
        plt.figure(figsize=(15, 10))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.show()
    else:
        print("Training curves not yet saved (training may still be running).")
else:
    print("No training runs found yet.")

In [ ]:
# @title 💾 Step 5 (Optional): Save best model to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
runs = sorted(glob.glob(f'{ROOT}/runs/*'))
if runs:
    latest_run = runs[-1]
    dest = '/content/drive/MyDrive/tactile_sdf_results'
    os.makedirs(dest, exist_ok=True)
    shutil.copytree(latest_run, f"{dest}/{os.path.basename(latest_run)}", dirs_exist_ok=True)
    print(f"✅ Results saved to Google Drive: {dest}")